# First match ≠ right match
## Relevance Scoring & Rerankers for Trustworthy AI + EU AI Act

**Author:** Ayhan Demir · Ironhack AI Consulting & Integration Bootcamp · Week 4 lab

**Goal:** build one retrieval path over two sources and prove that every answer is grounded in evidence a reviewer can inspect.

**Sources**
- `data/podcast.txt`: a podcast transcript about the EU *Ethics Guidelines for Trustworthy AI* (2019)
- `data/eu_ai_act.pdf`: Regulation (EU) 2024/1689, the AI Act (144 pages)

**Setup:** chunks of 1000 characters with 150 overlap · `text-embedding-3-small` · Pinecone serverless index `lab-rerankers` (cosine) · `gpt-4o-mini` at temperature 0. 

| Step | What I did | Result in one line |
|---|---|---|
| 0 | Environment and API keys | keys loaded from `.env` |
| 1 | Load, chunk with metadata, test embedding | 784 chunks (20 podcast, 764 Act), vectors of 1536 |
| 2 | Baseline vector search on 5 fixed test queries | right chunk for Q4 only at #3 |
| 3 | LLM relevance scoring as reranker (optional part) | order changed for 5 of 5 queries |
| 4 | Dedicated reranker | not implemented, I chose Step 3 |
| 5 | Metadata filtering, new schema, index rebuild | filters on section / article work, 112 of 113 articles tagged |
| 6 | RAG chain with citations, 1-argument switch | grounded answers, "I don't know" when out of scope |
| 7 | Manual evaluation with vs without reranking | better retrieval, same answers, 5x the time |
| 8 | Evidence for `lab_proof.md` + submission check | proof query Q4 (fines) |

**Reviewer entry point:** `lab_proof.md`.

---
## Step 0: Environment

My API keys live in a `.env` file next to this notebook (`OPENAI_API_KEY`, `PINECONE_API_KEY`); it is git-ignored and never committed. The two sources are in `data/`.
The first cell installs the packages, the second loads the keys and stops early if one is missing.

In [1]:
%pip install -U -q python-dotenv pypdf langchain-core langchain-community langchain-text-splitters langchain-openai pinecone langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

for key in ["OPENAI_API_KEY", "PINECONE_API_KEY"]:
    if not os.getenv(key):
        raise ValueError(f"{key} not found. Add it to your .env file.")
print("API keys loaded")

API keys loaded


---
## Step 1: Setup & Data Preparation

**Goal:** turn two very different sources into chunks that each carry **metadata**, meaning labels that say *where the chunk came from*.

1a loads both sources, 1b chunks them with a consistent metadata schema, 1c inspects the result, 1d tests the embedding model.

### 1a. Load both sources

In [3]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, TextLoader

DATA_DIR = Path("data")
PODCAST_PATH = DATA_DIR / "podcast.txt"
PDF_PATH = DATA_DIR / "eu_ai_act.pdf"

for path in [PODCAST_PATH, PDF_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Put the lab files in a 'data' folder next to this notebook.")

podcast_docs = TextLoader(str(PODCAST_PATH), encoding="utf-8").load()  # 1 Document (whole transcript)
pdf_docs = PyPDFLoader(str(PDF_PATH)).load()                           # 1 Document per PDF page

print(f"Podcast: {len(podcast_docs)} document, {len(podcast_docs[0].page_content):,} characters")
print(f"EU AI Act: {len(pdf_docs)} pages")
print("\nRaw metadata from the PDF loader:", pdf_docs[0].metadata)

/var/folders/v2/tdj6p3nj1bz3h2g88l897pr40000gn/T/ipykernel_24933/320955083.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


Podcast: 1 document, 16,414 characters
EU AI Act: 144 pages

Raw metadata from the PDF loader: {'producer': 'PDFlib+PDI 9.0.7p3 (C++/Win64)', 'creator': 'Servigistics Arbortext Publishing Engine', 'creationdate': '2024-07-11T14:47:17+02:00', 'author': 'Publications Office of the European Union L-2985 Luxembourg LUXEMBOURG', 'keywords': 'ISSN 1977-0677', 'moddate': '2024-07-11T15:55:28+02:00', 'subject': 'I Legislative acts', 'title': 'Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 2024 laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, (EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)Text with EEA relevance.', 'source': 'data/eu_ai_act.pdf', 'total_pages': 144, 'page': 0, 'page_label': '1'}


### 1b. Chunk with metadata

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

def chunk_with_metadata(docs, source_type, source_name):
    """Split documents into chunks and give every chunk a clean, consistent metadata schema."""
    chunks = splitter.split_documents(docs)
    for i, chunk in enumerate(chunks):
        metadata = {
            "source_type": source_type,        # "podcast" or "regulation" -> used for filtering in Step 5
            "source_name": source_name,        # human-readable name -> used for citations
            "chunk_id": f"{source_type}-{i}",  # unique id -> lets a reviewer trace an exact chunk
        }
        # PDF pages are 0-indexed in the loader; store the page number a human would see
        if "page" in chunk.metadata:
            metadata["page"] = int(chunk.metadata["page"]) + 1
        chunk.metadata = metadata
    return chunks

podcast_chunks = chunk_with_metadata(podcast_docs, "podcast", "Trustworthy AI podcast")
act_chunks = chunk_with_metadata(pdf_docs, "regulation", "EU AI Act (Reg. 2024/1689)")
all_chunks = podcast_chunks + act_chunks

print(f"Podcast chunks:   {len(podcast_chunks)}")
print(f"EU AI Act chunks: {len(act_chunks)}")
print(f"Total chunks:     {len(all_chunks)}")

Podcast chunks:   20
EU AI Act chunks: 764
Total chunks:     784


### 1c. Inspect the chunks
One chunk from each source, with its metadata. Looking at the data first is how I spotted that PDF chunks carry page headers (`OJ L, 12.7.2024`) inside the text.

In [5]:
for chunk in [podcast_chunks[0], act_chunks[150]]:
    print(chunk.metadata)
    print(chunk.page_content[:300], "...")
    print("-" * 80)

{'source_type': 'podcast', 'source_name': 'Trustworthy AI podcast', 'chunk_id': 'podcast-0'}
So imagine for a second you're driving across a, I don't know, a massive suspension bridge. Okay. You don't pull over halfway across, get out and demand to see the blueprints, right? You don't interview the welding crew. No, you just, you trust it. You just drive, you trust the bridge, you trust the ...
--------------------------------------------------------------------------------
{'source_type': 'regulation', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'chunk_id': 'regulation-150', 'page': 21}
indicators relating to AI.
OJ L, 12.7.2024 EN
ELI: http://data.europa.eu/eli/reg/2024/1689/oj 21/144
(35) Directive 2014/31/EU of the European Parliament and of the Council of 26 February 2014 on the harmonisation of the laws of the 
Member States relating to the making available on the market of no ...
--------------------------------------------------------------------------------


### 1d. Embeddings model

I use `OpenAIEmbeddings` with `text-embedding-3-small` (as in `rag_intro.ipynb`) and test it on **one** chunk. All chunks get embedded in Step 2, when they are uploaded to Pinecone, so I only pay for them once.

In [6]:
# Step 1d - embeddings model (rag_intro.ipynb, Section 2)
from langchain_openai import OpenAIEmbeddings

# 1. Create the embeddings object (reads OPENAI_API_KEY from the environment)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 2. Embed ONE chunk only, to check the model works (all chunks get embedded in Step 2)
test_vector = embeddings.embed_query(all_chunks[0].page_content)
print(f"Embedding dimension: {len(test_vector)}")   # expected: 1536
print(f"First 5 values: {test_vector[:5]}")

Embedding dimension: 1536
First 5 values: [0.028717041015625, 0.0396728515625, 0.00199127197265625, 0.039794921875, 0.0538330078125]



**Result:** 20 podcast chunks and 764 EU AI Act chunks, every chunk carries a metadata dict, and the test embedding has length **1536**.

---
## Step 2: Embeddings + Baseline Retrieval

**Goal:** a plain vector search as my **baseline**. Every later step has to beat it, or it isn't worth the extra cost.

What I do:
1. Create (or connect to) the Pinecone index `lab-rerankers`
2. Upload all chunks with `PineconeVectorStore.from_documents(...)`. 
3. `baseline_search(query, k)` returns `(doc, score)` pairs with `similarity_search_with_score`
4. Run it on five **fixed test queries**, the same ones in every step so the comparison is fair.

In [7]:
# Fixed test queries: the SAME five queries in every step, so the comparison is fair
TEST_QUERIES = [
    "What AI practices are prohibited under the EU AI Act?",
    "What are the seven requirements for trustworthy AI?",
    "What is the difference between human-in-the-loop and human-on-the-loop?",
    "What fines apply for using a prohibited AI practice?",
    "Is social scoring of citizens allowed?",
]

# Helper to print results in the same format in every step
def show_results(query, results, label="Results"):
    """results: list of (Document, score) pairs"""
    print(f"\n=== {label} | Query: {query}")
    for rank, (doc, score) in enumerate(results, 1):
        m = doc.metadata
        where = f"p.{m['page']}" if "page" in m else m.get("chunk_id")
        print(f"#{rank}  score={score:.3f}  [{m.get('source_type')} | {where}]  {doc.page_content[:120]!r}")

In [8]:
# create / connect to the Pinecone index and upload the chunks
import time
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index_name = "lab-rerankers"   # use lowercase letters and dashes only

# Create the index only if it does not exist yet
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1536,   # text-embedding-3-small
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"Created index: {index_name}")
else:
    print(f"Index {index_name} already exists")

# Upload (= embed + store) all chunks. Fixed ids make a re-run safe.
chunk_ids = [c.metadata["chunk_id"] for c in all_chunks]
vectorstore = PineconeVectorStore.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    index_name=index_name,
    ids=chunk_ids,
)

time.sleep(10)   # give Pinecone a moment to make the vectors searchable
stats = pc.Index(index_name).describe_index_stats()
print(f"Vectors in index: {stats.total_vector_count} (expected {len(all_chunks)})")

/opt/anaconda3/envs/bootcamp-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Index lab-rerankers already exists
Vectors in index: 784 (expected 784)


In [9]:
# baseline retrieval (rag_intro.ipynb, Section 4: similarity_search_with_score)
def baseline_search(query, k=5):
    """Plain vector search: returns (Document, cosine similarity) pairs, best first."""
    return vectorstore.similarity_search_with_score(query, k=k)

baseline_results = {}
for q in TEST_QUERIES:
    baseline_results[q] = baseline_search(q)
    show_results(q, baseline_results[q], label="Baseline")


=== Baseline | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.698  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a suffi'
#2  score=0.684  [regulation | p.12.0]  '(45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotec'
#3  score=0.681  [regulation | p.9.0]  'intention to distort behaviour where the distortion results from factors external to the AI system which are outside \nth'
#4  score=0.673  [regulation | p.45.0]  '(b) prohibitions of certain AI practices;\n(c) specific requirements for high-risk AI systems and obligations for operato'
#5  score=0.660  [regulation | p.3.0]  'operators involved in the AI value chain under this Regulation should apply without prejudice to national law, in \ncompl'

=== Baseline | Query: What are the seven requirements for trustworthy AI?
#1  score=0.674  [podcast | p

**Result:** each test query returns 5 results with a score, a source type and a page or chunk id.

---
## Step 3: LLM Relevance Scoring

**Goal:** let an LLM judge how relevant each retrieved chunk is, then re-order the results. This is my reranker.

What I do:
1. Retrieve a wider set of candidates (`k = 10`) with `baseline_search`.
2. Ask `gpt-4o-mini` to score each chunk's relevance to the query from 0 to 1.
3. Combine both scores, `combined = 0.3 * similarity + 0.7 * llm_score`, and sort by `combined`.
4. Compare the new top 3 with the baseline top 3.

Cost: 1 LLM call **per chunk, per query**, so 10 calls per query. I count them in `LLM_CALLS`.

In [10]:
# LLM relevance scoring
import re
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
LLM_CALLS = 0   # running counter of LLM calls (cost tracking for Step 7)

def llm_relevance_score(query, doc):
    """Ask the LLM how relevant one chunk is to the query. Returns a float between 0 and 1."""
    global LLM_CALLS
    prompt = (
        f"Rate the relevance of this document to the query '{query}' on a scale of 0-1.\n"
        "Reply with only the number, for example 0.7.\n\n"
        f"Document:\n{doc.page_content}"
    )
    response = llm.invoke(prompt)
    LLM_CALLS += 1
    match = re.search(r"\d+(?:\.\d+)?", response.content)   # first number in the reply
    try:
        score = float(match.group()) if match else 0.5
    except ValueError:
        score = 0.5                                          # neutral fallback, as in rag_pipeline.ipynb
    return min(max(score, 0.0), 1.0)                         # keep it inside 0..1

def llm_rerank(query, k_initial=10, k_final=5, w_sim=0.3, w_llm=0.7, verbose=False):
    """Retrieve k_initial chunks -> LLM-score each -> combine with similarity -> return top k_final."""
    candidates = baseline_search(query, k=k_initial)
    scored = []
    for doc, sim in candidates:
        llm_score = llm_relevance_score(query, doc)
        combined = w_sim * sim + w_llm * llm_score
        scored.append((doc, combined))
        if verbose:
            print(f"  {doc.metadata['chunk_id']:<16} sim={sim:.3f}  llm={llm_score:.2f}  combined={combined:.3f}")
    scored.sort(key=lambda pair: pair[1], reverse=True)
    return scored[:k_final]

In [11]:
# compare baseline top 3 vs LLM-reranked top 3 for every test query
llm_results = {}
calls_before = LLM_CALLS
for q in TEST_QUERIES:
    print(f"\n### {q}")
    llm_results[q] = llm_rerank(q, verbose=True)
    base_top3 = [d.metadata["chunk_id"] for d, _ in baseline_search(q, k=3)]
    llm_top3 = [d.metadata["chunk_id"] for d, _ in llm_results[q][:3]]
    print(f"Baseline top 3:     {base_top3}")
    print(f"LLM-reranked top 3: {llm_top3}")
    print("Order changed:", base_top3 != llm_top3)
    show_results(q, llm_results[q], label="LLM reranked")

print(f"\nLLM calls for this cell: {LLM_CALLS - calls_before} "
      f"({len(TEST_QUERIES)} queries x 10 chunks)")


### What AI practices are prohibited under the EU AI Act?
  regulation-342   sim=0.698  llm=1.00  combined=0.910
  regulation-84    sim=0.684  llm=0.30  combined=0.415
  regulation-58    sim=0.681  llm=0.60  combined=0.624
  regulation-313   sim=0.673  llm=0.50  combined=0.552
  regulation-15    sim=0.660  llm=0.20  combined=0.338
  regulation-83    sim=0.652  llm=0.80  combined=0.756
  regulation-685   sim=0.643  llm=0.10  combined=0.263
  regulation-59    sim=0.637  llm=0.60  combined=0.611
  regulation-317   sim=0.636  llm=0.20  combined=0.331
  regulation-54    sim=0.632  llm=0.90  combined=0.820
Baseline top 3:     ['regulation-342', 'regulation-84', 'regulation-58']
LLM-reranked top 3: ['regulation-342', 'regulation-54', 'regulation-83']
Order changed: True

=== LLM reranked | Query: What AI practices are prohibited under the EU AI Act?
#1  score=0.910  [regulation | p.51.0]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their b

**Result and my observations (Step 3)**

The order changed for all 5 queries (50 LLM calls for this cell).

| Query | Baseline top 3 | LLM-reranked top 3 | Better or worse? |
|---|---|---|---|
| Q1 prohibited practices | 342, 84, 58 | 342, 54, 313 | Better |
| Q2 seven requirements | podcast-2, 50, 151 | 50, podcast-2, 51 | Better |
| Q3 HITL vs HOTL | podcast-8, regulation-146, podcast-7 | podcast-8, podcast-7, podcast-18 | Better |
| Q4 fines | 342, 677, 665 | 665, 673, 677 | Clearly better|
| Q5 social scoring | 61, 60, 344 | 60, 344, podcast-16 | Mixed |

- With `w_llm = 0.7` the LLM score decides the order; similarity mostly breaks ties.
- The LLM only uses a few values (0.1, 0.2, ... 1.0), so ties happen (Q5: two chunks at 0.779).


---
## Step 4: Dedicated Reranker (optional, not implemented)

For the optional part I chose **Step 3 (LLM relevance scoring)** instead of a dedicated reranker (Cohere Rerank or a local cross-encoder such as `cross-encoder/ms-marco-MiniLM-L-6-v2`).

---
## Step 5: Metadata Filtering

**Goal:** restrict the search to the right part of the knowledge base *before* ranking (the lecture's "WHERE before WHICH").

What I do:
1. Filter by source with Pinecone's filter syntax, `filter={"source_type": {"$eq": "regulation"}}`, and run one query twice: regulation only, podcast only.
2. Try to filter on a field my metadata doesn't have yet (`article`). It returns nothing.
3. Add the missing fields to the chunking function, **delete the index and re-upload**, then filter again.
4. Show a filter that is too narrow, and the fix.


In [12]:
# filter by source_type (rag_pipeline.ipynb Section 4 idea, done inside Pinecone)
q = "Is social scoring of citizens allowed?"
for source in ["regulation", "podcast"]:
    results = vectorstore.similarity_search_with_score(q, k=5, filter={"source_type": {"$eq": source}})
    show_results(q, results, label=f"Filter source_type = {source}")


=== Filter source_type = regulation | Query: Is social scoring of citizens allowed?
#1  score=0.588  [regulation | p.9.0]  'multiple data points related to their social behaviour in multiple contexts or known, inferred or predicted personal \nor'
#2  score=0.497  [regulation | p.9.0]  'face or fingerprint, to deduce or infer an individuals’ political opinions, trade union membership, religious or \nphilos'
#3  score=0.497  [regulation | p.51.0]  '(c) the placing on the market, the putting into service or the use of AI systems for the evaluation or classification of'
#4  score=0.421  [regulation | p.16.0]  'should be granted, denied, reduced, revoked or reclaimed by authorities, including whether beneficiaries are \nlegitimate'
#5  score=0.412  [regulation | p.127.0]  '(a) AI systems intended to be used by public authorities or on behalf of public authorities to evaluate the eligibility '

=== Filter source_type = podcast | Query: Is social scoring of citizens allowed?
#1  score=0.485 

**The real lesson: try a field I don't have yet**
The next cell filters on `article`. My Step 1 metadata has no such key, so Pinecone finds **0** matches. Metadata can't be added at query time: it has to be written into the index when you upload.

In [13]:
# this filter can't work yet: the field does not exist in the index
results = vectorstore.similarity_search_with_score(
    "What does Article 5 say?", k=5, filter={"article": {"$eq": "Article 5"}}
)
print(f"Results with an 'article' filter: {len(results)}")   # expected: 0

Results with an 'article' filter: 0


### 5b. Better metadata schema (v2) and index rebuild

New keys (verified on this PDF: recitals p.1 to 43, `CHAPTER I` starts p.44, `ANNEX I` starts p.124):

| Key | Values | Used for |
|---|---|---|
| `section` | `recital`, `article`, `annex`, `transcript` (podcast) | "only articles", "only recitals" |
| `chapter` | `CHAPTER I` to `CHAPTER XIII` | broad topic filters |
| `article` | `Article 1` to `Article 113` | "What does Article 5 say?" |
| `annex` | `ANNEX I` to `ANNEX XIII` | "What is in Annex III?" |
| `char_start` | character position (podcast only) | trace a podcast chunk back to `podcast.txt` |

**How headings are detected:** a regex `^Article\s+(\d+)` on each line, matching the **whole line** (in-text references such as "Article 5(1) ..." are not headings). Chunks are in reading order, so the last heading seen is **carried forward** to the next chunks. A chunk gets the **first** heading it contains (or the carried one), then the carry is updated to its **last** heading.

**Known limit:** a chunk gets ONE article tag, the first heading inside it. Text before that heading (the tail of the previous article) or after a second heading is tagged wrongly. Examples from my run: `regulation-342` holds Article 4 and the start of Article 5(1) but is tagged `Article 4`; `regulation-313` holds Article 1(2) but is tagged `Article 2`.
**Pinecone rule:** metadata values can't be `None`, so missing keys are simply left out.

In [14]:
# chunking v2 with a richer metadata schema
RECITALS_LAST_PAGE = 43    # recitals end here
ANNEX_FIRST_PAGE = 124     # "ANNEX I" starts here

ARTICLE_RE = re.compile(r"^Article\s+(\d+)\s*$")
CHAPTER_RE = re.compile(r"^CHAPTER\s+([IVX]+)\s*$")
ANNEX_RE = re.compile(r"^ANNEX\s+([IVX]+)\s*$")

# add_start_index=True stores each chunk's character position (used for the podcast)
splitter_v2 = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150, add_start_index=True)

def find_headings(text, pattern):
    """Return every heading value (e.g. '5' for 'Article 5') found on its own line."""
    headings = []
    for line in text.splitlines():
        match = pattern.match(line.strip())
        if match:
            headings.append(match.group(1))
    return headings

def chunk_with_metadata_v2(docs, source_type, source_name):
    """Same as chunk_with_metadata, plus section / chapter / article / annex / char_start."""
    chunks = splitter_v2.split_documents(docs)
    current = {}   # last heading seen so far, carried forward from chunk to chunk
    for i, chunk in enumerate(chunks):
        old = chunk.metadata
        metadata = {
            "source_type": source_type,
            "source_name": source_name,
            "chunk_id": f"{source_type}-{i}",
        }
        if source_type == "podcast":
            metadata["section"] = "transcript"
            metadata["char_start"] = int(old["start_index"])
        else:
            page = int(old["page"]) + 1
            metadata["page"] = page
            if page <= RECITALS_LAST_PAGE:
                metadata["section"] = "recital"
            else:
                if page < ANNEX_FIRST_PAGE:
                    metadata["section"] = "article"
                    keys = [("chapter", "CHAPTER", CHAPTER_RE), ("article", "Article", ARTICLE_RE)]
                else:
                    metadata["section"] = "annex"
                    keys = [("annex", "ANNEX", ANNEX_RE)]
                for key, label, pattern in keys:
                    headings = find_headings(chunk.page_content, pattern)
                    value = headings[0] if headings else current.get(key)
                    if value:                       # leave the key out instead of storing None
                        metadata[key] = f"{label} {value}"
                    if headings:
                        current[key] = headings[-1]
        chunk.metadata = metadata
    return chunks

podcast_chunks = chunk_with_metadata_v2(podcast_docs, "podcast", "Trustworthy AI podcast")
act_chunks = chunk_with_metadata_v2(pdf_docs, "regulation", "EU AI Act (Reg. 2024/1689)")
all_chunks = podcast_chunks + act_chunks

from collections import Counter
print(f"Total chunks: {len(all_chunks)}")
print("Sections:", Counter(c.metadata["section"] for c in all_chunks))
print("Articles tagged:", len({c.metadata["article"] for c in act_chunks if "article" in c.metadata}), "of 113")
print("Example:", next(c.metadata for c in act_chunks if c.metadata.get("article") == "Article 5"))

Total chunks: 784
Sections: Counter({'article': 396, 'recital': 308, 'annex': 60, 'transcript': 20})
Articles tagged: 112 of 113
Example: {'source_type': 'regulation', 'source_name': 'EU AI Act (Reg. 2024/1689)', 'chunk_id': 'regulation-343', 'page': 51, 'section': 'article', 'chapter': 'CHAPTER II', 'article': 'Article 5'}


**Rebuild the index.** The old vectors carry the old metadata, so I delete the index and upload again. After this cell, `vectorstore` (and therefore `baseline_search` and `llm_rerank`) uses the new index automatically.

In [15]:
# delete + recreate the index, re-upload with the v2 metadata
pc.delete_index(index_name)
while index_name in pc.list_indexes().names():   # wait until the delete has finished
    time.sleep(2)

pc.create_index(
    name=index_name,
    dimension=1536,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)

chunk_ids = [c.metadata["chunk_id"] for c in all_chunks]
vectorstore = PineconeVectorStore.from_documents(
    documents=all_chunks, embedding=embeddings, index_name=index_name, ids=chunk_ids
)
time.sleep(10)
stats = pc.Index(index_name).describe_index_stats()
print(f"Vectors in rebuilt index: {stats.total_vector_count} (expected {len(all_chunks)})")

Vectors in rebuilt index: 784 (expected 784)


In [16]:
# filter on the NEW metadata fields
def show_filtered(query, flt, label):
    """Run a filtered search, print it, and check every result really matches the filter."""
    results = vectorstore.similarity_search_with_score(query, k=5, filter=flt)
    show_results(query, results, label=label)
    for doc, _ in results:
        m = doc.metadata
        print(f"    {m['chunk_id']:<16} section={m.get('section')}  article={m.get('article', '-')}")
    return results

# 1. Only Article 5 chunks
show_filtered("What does Article 5 say?", {"article": {"$eq": "Article 5"}}, "Filter article = Article 5")

# 2. Only recitals
show_filtered("Is social scoring of citizens allowed?", {"section": {"$eq": "recital"}}, "Filter section = recital")

# 3. Combine two conditions: regulation AND article text (no recitals, no podcast)
_ = show_filtered(
    "Is social scoring of citizens allowed?",
    {"$and": [{"source_type": {"$eq": "regulation"}}, {"section": {"$eq": "article"}}]},
    "Filter regulation AND section = article",
)


=== Filter article = Article 5 | Query: What does Article 5 say?
#1  score=0.273  [regulation | p.53.0]  'a minimum, contain the information specified under paragraph 6 and shall not include sensitive operational data.\n5. A Me'
#2  score=0.257  [regulation | p.52.0]  'beings, as well as the search for missing persons;\n(ii) the prevention of a specific, substantial and imminent threat to'
#3  score=0.257  [regulation | p.52.0]  'In addition, the use of ‘real-time’ remote biometric identification systems in publicly accessible spaces for the purpos'
#4  score=0.251  [regulation | p.52.0]  '(g) the placing on the market, the putting into service for this specific purpose, or the use of biometric categorisatio'
#5  score=0.245  [regulation | p.53.0]  'objectives specified in paragraph 1, first subparagraph, point (h), as identified in the request and, in particular, rem'
    regulation-354   section=article  article=Article 5
    regulation-348   section=article  article=Article 5
    r

**Too-narrow filter (lecture: "Germany vs EU").** "Prohibited practice" makes you think of Article 5, but the **fines** are in Article 99. A filter on Article 5 alone hides the answer; widening it with `$in` fixes that.

In [17]:
# a filter that is too narrow, and the fix
q = "What fines apply for using a prohibited AI practice?"
show_filtered(q, {"article": {"$eq": "Article 5"}}, "TOO NARROW: article = Article 5")
_ = show_filtered(q, {"article": {"$in": ["Article 5", "Article 99"]}}, "FIXED: article in [Article 5, Article 99]")


=== TOO NARROW: article = Article 5 | Query: What fines apply for using a prohibited AI practice?
#1  score=0.552  [regulation | p.51.0]  'materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an '
#2  score=0.512  [regulation | p.51.0]  '(c) the placing on the market, the putting into service or the use of AI systems for the evaluation or classification of'
#3  score=0.473  [regulation | p.51.0]  'assessments of natural persons in order to assess or predict the risk of a natural person committing a criminal offence,'
#4  score=0.430  [regulation | p.51.0]  'of a natural person in the areas of workplace and education institutions, except where the use of the AI system is \ninte'
#5  score=0.359  [regulation | p.52.0]  '(g) the placing on the market, the putting into service for this specific purpose, or the use of biometric categorisatio'
    regulation-343   section=article  article=Article 5
    regulation-344   section=

**Result and my observations (Step 5)**
- **Filter by source:** same query, two worlds. `regulation` returns recital (31) and Article 5(1)(c) on social scoring; `podcast` returns the "citizen scoring" passage (`podcast-16`) with lower scores (0.31 to 0.49).
- **Missing field:** before the rebuild the `article` filter returned **0** results. Pinecone can only filter on metadata that was written at upload time.
- **Why I had to rebuild:** metadata is stored *with each vector*. Changing `chunk_with_metadata` doesn't touch what's already in the index, so I deleted the index and re-uploaded: 784 vectors, 112 of 113 articles tagged.
- **New filters work:** `article = Article 5` returns only Article 5 chunks. Their scores are low (0.25 to 0.27) because "What does Article 5 say?" carries almost no meaning for the embedding; the filter does the work. `regulation AND section = article` puts Article 5(1)(c) (`regulation-344`) at #1 for social scoring.
- **Too-narrow filter:** `Article 5` alone returns five prohibition chunks and no fines. `$in [Article 5, Article 99]` brings Article 99(3) (EUR 35 000 000 / 7 %) to #1. Same lesson as the lecture's "Germany vs EU": the filter did exactly what I told it.
- **Side effect:** Pinecone returns numbers as floats, so pages print as `p.51.0`. Cosmetic only; `int(page)` in the label would fix it.

---
## Step 6: Complete RAG Pipeline with Reranking

**Goal:** plug the retrieval into a full RAG chain that **cites its sources**.

What I do:
1. An LCEL chain `retrieve → format context → prompt → LLM → answer` (`rag_intro.ipynb`, Section 5).
2. `format_context` puts each chunk's source, page and article **inside** the context, and the prompt tells the LLM to cite them.
3. If the context doesn't contain the answer, the model must say *"I don't know based on the sources."*

In [18]:
# RAG pipeline with citations
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def source_label(doc):
    """Human-readable citation label for one chunk."""
    m = doc.metadata
    if "page" in m:
        label = f"{m['source_name']}, p.{m['page']}"
        if "article" in m:
            label += f", {m['article']}"
        elif "annex" in m:
            label += f", {m['annex']}"
        return label
    return f"{m['source_name']}, {m['chunk_id']}"

def format_context(results):
    """Join chunks into one string, each prefixed with its source label."""
    return "\n\n".join(f"[{source_label(doc)}]\n{doc.page_content}" for doc, _ in results)

rag_prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context.
After each claim, cite the source label in square brackets exactly as it appears in the context.
If the context does not contain the answer, reply exactly: "I don't know based on the sources."

Context:
{context}

Question: {question}

Answer:"""
)

rag_chain = rag_prompt | llm | StrOutputParser()   # LCEL: prompt -> LLM -> plain text

def answer(query, retrieve_fn=baseline_search):   # <- swap retrieve_fn to switch methods
    """Retrieve -> format -> prompt | llm | parser. Returns (answer_text, results_used)."""
    global LLM_CALLS
    results = retrieve_fn(query)
    text = rag_chain.invoke({"context": format_context(results), "question": query})
    LLM_CALLS += 1   # the answer itself is one more LLM call
    return text, results

In [19]:
# one query, both methods. The ONLY difference is retrieve_fn.
q = TEST_QUERIES[3]   # "What fines apply for using a prohibited AI practice?"

base_answer, base_used = answer(q)                          # baseline
llm_answer, llm_used = answer(q, retrieve_fn=llm_rerank)    # LLM-reranked (1-line change)

for label, text, used in [("Baseline", base_answer, base_used), ("LLM reranked", llm_answer, llm_used)]:
    print(f"\n=== {label}\n{text}\nChunks used:")
    for doc, score in used:
        print(f"  [{source_label(doc)}]  score={score:.3f}")


=== Baseline
Non-compliance with the prohibition of the AI practices referred to in Article 5 shall be subject to administrative fines of up to EUR 35,000,000 or, if the offender is an undertaking, up to 7% of its total worldwide annual turnover for the preceding financial year, whichever is higher [EU AI Act (Reg. 2024/1689), p.115.0, Article 99].
Chunks used:
  [EU AI Act (Reg. 2024/1689), p.51.0, Article 4]  score=0.634
  [EU AI Act (Reg. 2024/1689), p.118.0, Article 101]  score=0.617
  [EU AI Act (Reg. 2024/1689), p.115.0, Article 99]  score=0.613
  [EU AI Act (Reg. 2024/1689), p.117.0, Article 100]  score=0.610
  [EU AI Act (Reg. 2024/1689), p.117.0, Article 101]  score=0.584

=== LLM reranked
Non-compliance with the prohibition of the AI practices referred to in Article 5 shall be subject to administrative fines of up to EUR 35,000,000 or, if the offender is an undertaking, up to 7% of its total worldwide annual turnover for the preceding financial year, whichever is higher [EU 

In [20]:
# grounding checks
# 1. Out-of-scope question: the pipeline should refuse
oos_answer, _ = answer("What is the capital of France?")
print("Out of scope ->", oos_answer)

# 2. Optional: a filtered retriever plugs in the same way (regulation only)
def regulation_only(query, k=5):
    return vectorstore.similarity_search_with_score(query, k=k, filter={"source_type": {"$eq": "regulation"}})

reg_answer, _ = answer("Is social scoring of citizens allowed?", retrieve_fn=regulation_only)
print("\nRegulation only ->", reg_answer)

Out of scope -> I don't know based on the sources.

Regulation only -> No, social scoring of citizens is prohibited if it leads to detrimental or unfavourable treatment of natural persons or groups based on their social behaviour or personal characteristics. [EU AI Act (Reg. 2024/1689), p.51.0, Article 5]


**Result and my observations (Step 6)**
- **Q4:** both methods give the same correct answer and cite Article 99, p.115. The difference is in the evidence: the baseline found the right chunk at rank 3, the reranked pipeline at rank 1.
- **Out of scope:** "What is the capital of France?" returns *"I don't know based on the sources."* The refusal instruction works.
- **Regulation-only retriever:** switching `retrieve_fn` to a metadata filter (no extra LLM calls) makes the social-scoring answer cite **Article 5**, p.51, instead of a recital.

---
## Step 7: Evaluate With vs. Without Reranking

**Goal:** decide, with evidence, whether reranking was worth it.

I run every test query through both pipelines, time them and count the LLM calls. Then I read each answer and the chunks behind it myself and judge three things: is the top-1 chunk relevant, is the answer correct, and is it grounded in the chunk it cites. No automatic metric: I am the judge.

In [21]:
# baseline vs LLM-reranked answers side by side, timed, with LLM-call counts
eval_rows = []
for q in TEST_QUERIES:
    print(f"\n{'=' * 100}\nQUERY: {q}")
    for method, fn in [("Baseline", baseline_search), ("LLM reranked", llm_rerank)]:
        calls_before = LLM_CALLS
        start = time.time()
        text, used = answer(q, retrieve_fn=fn)
        seconds = time.time() - start
        calls = LLM_CALLS - calls_before
        eval_rows.append({"query": q, "method": method, "top1": source_label(used[0][0]),
                          "seconds": seconds, "calls": calls})
        print(f"\n--- {method}  ({seconds:.1f}s, {calls} LLM calls)\n{text}")
        for doc, score in used[:3]:
            print(f"    [{source_label(doc)}]  score={score:.3f}")


QUERY: What AI practices are prohibited under the EU AI Act?

--- Baseline  (1.8s, 1 LLM calls)
The following AI practices are prohibited under the EU AI Act: the placing on the market, the putting into service, or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an informed decision [EU AI Act (Reg. 2024/1689), p.51.0, Article 5].
    [EU AI Act (Reg. 2024/1689), p.51.0, Article 4]  score=0.698
    [EU AI Act (Reg. 2024/1689), p.12.0]  score=0.684
    [EU AI Act (Reg. 2024/1689), p.9.0]  score=0.681

--- LLM reranked  (9.1s, 11 LLM calls)
The following AI practices are prohibited under the EU AI Act: 

1. The placing on the market, the putting into service, or the use of an AI system that deploys subliminal techniques beyond a person’s co

### My evaluation

I read every answer and the chunks behind it, and checked the claims against the Act text.

| Query | Method | Top-1 chunk | Top-1 relevant? | Answer correct? | Grounded in cited chunk? | Time (s) | LLM calls | Notes |
|---|---|---|---|---|---|---|---|---|
| Q1 prohibited practices | Baseline | p.51, `342` (Art 4 + start of Art 5) | Partial | Partial | Y | 2.1 | 1 | Lists only Art 5(1)(a); the other 7 prohibitions are missing. Cites "Article 5" although the context label said "Article 4" |
| Q1 | Reranked | p.51, `342` | Partial | Partial | Partial | 10.4 | 11 | Adds emotion recognition at work (correct, recital 44). Claim 3 is **wrong**: practices banned by *other* Union law (recital 45) are presented as AI Act prohibitions |
| Q2 seven requirements | Baseline | `podcast-2` | N | N | n/a | 1.1 | 1 | "I don't know", although recital (27) with all seven was #2 in the context |
| Q2 | Reranked | p.8, `50` (recital 27) | Y | N | n/a | 8.3 | 11 | Right chunk at #1, still "I don't know". The Act calls them "seven ethical **principles**", my query said "requirements" |
| Q3 HITL vs HOTL | Baseline | `podcast-8` | Y | Y | Y | 2.0 | 1 | Correct, from the podcast only (the Act doesn't use these terms) |
| Q3 | Reranked | `podcast-8` | Y | Y | Y | 8.5 | 11 | Same answer, less noise in the context |
| Q4 fines | Baseline | p.51, `342` | N | Y | Y | 1.7 | 1 | Right chunk only at #3; the LLM still found it. EUR 35M or 7 % |
| Q4 | Reranked | p.115, `665` (Art 99) | Y | Y | Y | 9.6 | 11 | Same answer, right evidence first. Both miss Art 100(2) (EUR 1.5M for EU institutions), which was in the context |
| Q5 social scoring | Baseline | p.9, `61` (recital 31) | Y | Y | Y | 1.6 | 1 | Cites Article 5 |
| Q5 | Reranked | p.9, `60` (recitals 30/31) | Y | Y | Y | 8.9 | 11 | Cites a recital instead of the Article; a podcast chunk is in the top 3 |

**Summary**

| | Baseline | LLM reranked |
|---|---|---|
| Top-1 chunk relevant | 2 Y, 1 Partial, 2 N | 4 Y, 1 Partial |
| Answer correct | 3 Y, 1 Partial, 1 N | 3 Y, 1 Partial, 1 N |
| Avg time per query | 1.7 s | 9.1 s |
| LLM calls per query | 1 | 11 |

**Did reranking change the answer, or only the order?**
Mostly only the order. Retrieval clearly got better (top-1 relevant from 2 to 4 of 5), but no answer went from wrong to right:
- Q4: the baseline top 5 already contained Article 99, so the answer model found it anyway.
- Q2: retrieval was fixed, but the answer still failed. That's a generation problem (wording "requirements" vs "principles" plus a strict "only from context" prompt), not a ranking problem.
- Q1: the reranked answer got longer and picked up one unsupported claim. More context gives more room to misread.

**What did it cost, and would a client pay?**
11 LLM calls instead of 1 and about 5x the time (9.1 s vs 1.7 s per query). For this setup I wouldn't sell it as is. It pays off when the context is small (k = 2 to 3), when the corpus is large enough that the right chunk falls outside the top k, or as a precision step for high-stakes legal questions. Here, the cheaper win was the **metadata filter**: "regulation only" gave the more authoritative Article 5 citation at zero extra calls. Next I would test a dedicated reranker (Step 4) for latency, and a query rewrite "requirements → principles" for Q2.

---
## Step 8: Proof and Submission Check


**Evidence printout:** runs one query through both methods and prints the evidence as markdown tables (here Q1). I used the same printout format for the Q4 tables in `lab_proof.md`.

In [22]:
# evidence printout for lab_proof.md
PROOF_QUERY = TEST_QUERIES[0]   # the query to print evidence for

for method, fn in [("Baseline", baseline_search), ("LLM reranked", llm_rerank)]:
    text, used = answer(PROOF_QUERY, retrieve_fn=fn)
    print(f"\n## {method}\n**Query:** {PROOF_QUERY}\n")
    print("| Rank | Source | Page / chunk_id | Section / Article | Score | Snippet |")
    print("|---|---|---|---|---|---|")
    for rank, (doc, score) in enumerate(used, 1):
        m = doc.metadata
        where = f"p.{m['page']}" if "page" in m else m["chunk_id"]
        part = m.get("article") or m.get("annex") or m.get("section", "")
        snippet = doc.page_content[:100].replace("\n", " ").replace("|", "/")
        print(f"| {rank} | {m['source_name']} | {where} | {part} | {score:.3f} | {snippet}... |")
    print(f"\n**Answer:**\n\n{text}")


## Baseline
**Query:** What AI practices are prohibited under the EU AI Act?

| Rank | Source | Page / chunk_id | Section / Article | Score | Snippet |
|---|---|---|---|---|---|
| 1 | EU AI Act (Reg. 2024/1689) | p.51.0 | Article 4 | 0.698 | Article 4 AI literacy Providers and deployers of AI systems shall take measures to ensure, to their ... |
| 2 | EU AI Act (Reg. 2024/1689) | p.12.0 | recital | 0.684 | (45) Practices that are prohibited by Union law, including data protection law, non-discrimination l... |
| 3 | EU AI Act (Reg. 2024/1689) | p.9.0 | recital | 0.682 | intention to distort behaviour where the distortion results from factors external to the AI system w... |
| 4 | EU AI Act (Reg. 2024/1689) | p.45.0 | Article 2 | 0.673 | (b) prohibitions of certain AI practices; (c) specific requirements for high-risk AI systems and obl... |
| 5 | EU AI Act (Reg. 2024/1689) | p.3.0 | recital | 0.660 | operators involved in the AI value chain under this Regulation should apply without p

## Baseline
**Query:** What AI practices are prohibited under the EU AI Act?

| Rank | Source | Page / chunk_id | Section / Article | Score | Snippet |
|---|---|---|---|---|---|
| 1 | EU AI Act (Reg. 2024/1689) | p.51.0 | Article 4 | 0.698 | Article 4 AI literacy Providers and deployers of AI systems shall take measures to ensure, to their ... |
| 2 | EU AI Act (Reg. 2024/1689) | p.12.0 | recital | 0.684 | (45) Practices that are prohibited by Union law, including data protection law, non-discrimination l... |
| 3 | EU AI Act (Reg. 2024/1689) | p.9.0 | recital | 0.682 | intention to distort behaviour where the distortion results from factors external to the AI system w... |
| 4 | EU AI Act (Reg. 2024/1689) | p.45.0 | Article 2 | 0.673 | (b) prohibitions of certain AI practices; (c) specific requirements for high-risk AI systems and obl... |
| 5 | EU AI Act (Reg. 2024/1689) | p.3.0 | recital | 0.660 | operators involved in the AI value chain under this Regulation should apply without prejudice to nat... |

**Answer:**

The following AI practices are prohibited under the EU AI Act: the placing on the market, the putting into service, or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an informed decision [EU AI Act (Reg. 2024/1689), p.51.0, Article 5].

## LLM reranked
**Query:** What AI practices are prohibited under the EU AI Act?

| Rank | Source | Page / chunk_id | Section / Article | Score | Snippet |
|---|---|---|---|---|---|
| 1 | EU AI Act (Reg. 2024/1689) | p.51.0 | Article 4 | 0.910 | Article 4 AI literacy Providers and deployers of AI systems shall take measures to ensure, to their ... |
| 2 | EU AI Act (Reg. 2024/1689) | p.8.0 | recital | 0.820 | manipulative, exploitative and social control practices. Such practices are particularly harmful and... |
| 3 | EU AI Act (Reg. 2024/1689) | p.12.0 | recital | 0.756 | generalisability. Therefore, AI systems identifying or inferring emotions or intentions of natural p... |
| 4 | EU AI Act (Reg. 2024/1689) | p.9.0 | recital | 0.624 | intention to distort behaviour where the distortion results from factors external to the AI system w... |
| 5 | EU AI Act (Reg. 2024/1689) | p.9.0 | recital | 0.611 | irrespective of whether they are put in place through AI systems or otherwise. The prohibitions of m... |

**Answer:**

The following AI practices are prohibited under the EU AI Act: 

1. The placing on the market, the putting into service, or the use of an AI system that deploys subliminal techniques beyond a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make decisions [EU AI Act (Reg. 2024/1689), p.51.0, Article 5].

2. AI systems intended to be used to detect the emotional state of individuals in situations related to the workplace and education, except for those placed on the market strictly for medical or safety reasons [EU AI Act (Reg. 2024/1689), p.12.0].

3. Practices that lead to significant harms, particularly those that have sufficiently important adverse impacts on physical, psychological health, or financial interests, are also prohibited [EU AI Act (Reg. 2024/1689), p.8.0]. 

4. Manipulative and exploitative practices that result in economic or financial harms to consumers are prohibited under all circumstances, irrespective of whether they are implemented through AI systems or otherwise [EU AI Act (Reg. 2024/1689), p.9.0].